# 🧹 CleanFlow AI Agent
**ITAI 2376 – Deep Learning in Artificial Intelligence | Spring 2026**  
**Sharise Griggs | Solo Project**

CleanFlow is a smart cleaning assistant that takes a user's natural language description of their home situation and generates a personalized, prioritized cleaning plan they can actually follow.

---

## Cell 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install langchain langchain-openai langchain-community openai transformers torch sentence-transformers faiss-cpu python-dotenv --quiet

## Cell 2: Imports and Environment Setup

In [ ]:
import os
import json
import re
from dotenv import load_dotenv

# LangChain core imports
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage

# Transformer / embedding imports for deep learning connection
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("❌ OPENAI_API_KEY not found. Please set it in your .env file.")

print("✅ Environment loaded successfully.")

## Cell 3: CleanFlow Knowledge Base (Task Dataset)
This dataset was designed for the blueprint and represents the structured knowledge the agent uses for task lookup and time estimation.

In [ ]:
# ---------------------------------------------------------------
# CLEANFLOW TASK KNOWLEDGE BASE
# Each entry: task name, room, urgency level, estimated minutes
# This dataset is referenced by the scheduling tool
# ---------------------------------------------------------------

TASK_KNOWLEDGE_BASE = [
    {"task": "wash dishes",         "room": "kitchen",       "urgency": "high",   "minutes": 10},
    {"task": "wipe counters",        "room": "kitchen",       "urgency": "medium", "minutes": 8},
    {"task": "clean stovetop",       "room": "kitchen",       "urgency": "medium", "minutes": 10},
    {"task": "take out trash",       "room": "kitchen",       "urgency": "high",   "minutes": 5},
    {"task": "mop kitchen floor",    "room": "kitchen",       "urgency": "low",    "minutes": 15},
    {"task": "sweep floor",          "room": "living room",   "urgency": "medium", "minutes": 12},
    {"task": "vacuum carpet",        "room": "living room",   "urgency": "medium", "minutes": 15},
    {"task": "dust surfaces",        "room": "living room",   "urgency": "low",    "minutes": 10},
    {"task": "organize clutter",     "room": "living room",   "urgency": "medium", "minutes": 20},
    {"task": "clean toilet",         "room": "bathroom",      "urgency": "high",   "minutes": 7},
    {"task": "scrub sink",           "room": "bathroom",      "urgency": "medium", "minutes": 8},
    {"task": "wipe mirror",          "room": "bathroom",      "urgency": "low",    "minutes": 5},
    {"task": "scrub shower",         "room": "bathroom",      "urgency": "medium", "minutes": 15},
    {"task": "do laundry (start)",   "room": "laundry room",  "urgency": "high",   "minutes": 5},
    {"task": "fold laundry",         "room": "laundry room",  "urgency": "medium", "minutes": 15},
    {"task": "vacuum bedroom",       "room": "bedroom",       "urgency": "low",    "minutes": 15},
    {"task": "make bed",             "room": "bedroom",       "urgency": "medium", "minutes": 5},
    {"task": "change bed sheets",    "room": "bedroom",       "urgency": "low",    "minutes": 10},
    {"task": "organize closet",      "room": "bedroom",       "urgency": "low",    "minutes": 30},
    {"task": "wipe baseboards",      "room": "general",       "urgency": "low",    "minutes": 20},
    {"task": "clean windows",        "room": "general",       "urgency": "low",    "minutes": 20},
]

# Urgency ranking for sorting (higher = more urgent)
URGENCY_RANK = {"high": 3, "medium": 2, "low": 1}

print(f"✅ Task knowledge base loaded — {len(TASK_KNOWLEDGE_BASE)} tasks available.")

## Cell 4: Deep Learning Layer — BERT-Based Task Extractor
This is the Transformer (BERT) component from the blueprint. It uses sentence embeddings to match user input against known cleaning tasks using cosine similarity — this is real semantic understanding, not keyword matching.

In [ ]:
# ---------------------------------------------------------------
# DEEP LEARNING COMPONENT 1: BERT-Based Semantic Task Extractor
# Uses sentence-transformers (BERT backbone) to embed user input
# and match it against our task knowledge base via cosine similarity.
# This is the Transformer module connection from the blueprint.
# ---------------------------------------------------------------

print("⏳ Loading BERT embedding model (sentence-transformers)...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight BERT-based model

# Pre-compute embeddings for all tasks in the knowledge base
task_descriptions = [entry["task"] for entry in TASK_KNOWLEDGE_BASE]
task_embeddings = embedding_model.encode(task_descriptions, convert_to_tensor=True)

def extract_tasks_with_bert(user_input: str, top_k: int = 8, threshold: float = 0.25) -> list:
    """
    Uses BERT-based sentence embeddings and cosine similarity to identify
    which cleaning tasks are semantically relevant to the user's input.
    
    This is the Transformer connection from the blueprint:
    the self-attention mechanism in BERT understands context across
    the full sentence, not just individual keywords.
    
    Args:
        user_input: Raw natural language input from the user
        top_k: Maximum tasks to return
        threshold: Minimum cosine similarity score to include a task
    
    Returns:
        List of matching task dicts sorted by similarity score
    """
    # Embed the user's input using the same BERT model
    input_embedding = embedding_model.encode(user_input, convert_to_tensor=True)
    
    # Compute cosine similarity between input and all task embeddings
    cos_scores = torch.nn.functional.cosine_similarity(
        input_embedding.unsqueeze(0), task_embeddings
    )
    
    # Get top_k matches above the threshold
    scores = cos_scores.cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    matched_tasks = []
    for idx in top_indices:
        if scores[idx] >= threshold:
            task_entry = TASK_KNOWLEDGE_BASE[idx].copy()
            task_entry["similarity_score"] = round(float(scores[idx]), 3)
            matched_tasks.append(task_entry)
    
    return matched_tasks


def extract_time_available(user_input: str) -> int:
    """
    Extracts time constraint from user input using regex pattern matching.
    Returns estimated available minutes (defaults to 45 if none found).
    """
    # Look for patterns like '30 minutes', '1 hour', '2 hrs', 'half an hour'
    patterns = [
        r'(\d+)\s*(?:minute|min)',
        r'(\d+)\s*(?:hour|hr)',
        r'half\s*an?\s*hour',
        r'an?\s*hour'
    ]
    
    text = user_input.lower()
    
    match = re.search(r'(\d+)\s*(?:minute|min)', text)
    if match:
        return int(match.group(1))
    
    match = re.search(r'(\d+)\s*(?:hour|hr)', text)
    if match:
        return int(match.group(1)) * 60
    
    if 'half an hour' in text or 'half hour' in text:
        return 30
    
    if 'an hour' in text or 'one hour' in text:
        return 60
    
    # No time found — default to 45 minutes
    return 45


print("✅ BERT semantic task extractor ready.")

# Quick test to confirm the embedding model works
test_result = extract_tasks_with_bert("my kitchen is a mess and dishes are piling up")
print(f"\n🧪 Test — 'my kitchen is a mess and dishes are piling up':")
for t in test_result[:3]:
    print(f"   → {t['task']} ({t['room']}, {t['urgency']}) | similarity: {t['similarity_score']}")

## Cell 5: Sequence Logic Layer (RNN-Inspired Task Ordering)
This is the second deep learning connection from the blueprint — RNN sequence modeling concepts applied to task ordering.

In [ ]:
# ---------------------------------------------------------------
# DEEP LEARNING COMPONENT 2: Sequence-Based Task Ordering
# Inspired by RNN sequence modeling from the course module.
# Cleaning is inherently sequential — you can't mop before sweeping.
# We model the session as a time series where each task has:
#   - a position in the sequence
#   - dependencies on prior tasks
#   - a duration that affects what fits next
# ---------------------------------------------------------------

# Dependency rules: task X must come before task Y
# Models the "hidden state" concept from RNNs — previous outputs affect next steps
TASK_DEPENDENCIES = {
    "mop kitchen floor":   ["sweep floor", "wipe counters"],   # sweep before mop
    "fold laundry":        ["do laundry (start)"],              # start before fold
    "vacuum carpet":       ["organize clutter"],                # declutter before vacuuming
    "vacuum bedroom":      ["make bed"],                        # make bed first
    "scrub shower":        ["scrub sink"],                      # sink before shower typically
}

# Room priority sequence: which rooms are tackled first
# This is the "time series order" concept — position matters
ROOM_SEQUENCE_PRIORITY = {
    "kitchen":      1,   # Food hygiene — highest priority
    "bathroom":     2,   # Health/sanitation
    "laundry room": 3,   # Start machines early so they run while you clean
    "living room":  4,   # Common area
    "bedroom":      5,   # Private space
    "general":      6,   # Least urgent
}

def sequence_order_tasks(tasks: list, time_available: int) -> list:
    """
    Orders and filters tasks using RNN-inspired sequence logic.
    
    Applies three sequential passes (like RNN processing steps):
      Pass 1: Sort by urgency (hidden state: what is most pressing)
      Pass 2: Apply room sequence ordering (temporal position matters)
      Pass 3: Fit tasks within the available time budget (like a sliding window)
    
    Args:
        tasks: List of matched task dicts from the BERT extractor
        time_available: Minutes the user has available
    
    Returns:
        Ordered list of tasks that fit within the time budget
    """
    if not tasks:
        return []
    
    # Pass 1 — Sort by urgency then room sequence priority
    sorted_tasks = sorted(
        tasks,
        key=lambda t: (
            -URGENCY_RANK.get(t["urgency"], 1),           # urgency descending
            ROOM_SEQUENCE_PRIORITY.get(t["room"], 99)      # room sequence ascending
        )
    )
    
    # Pass 2 — Apply dependency ordering (like RNN hidden state carrying context forward)
    ordered = []
    task_names = [t["task"] for t in sorted_tasks]
    
    for task in sorted_tasks:
        deps = TASK_DEPENDENCIES.get(task["task"], [])
        # Move dependencies before this task if they are in the list
        for dep_name in deps:
            dep_task = next((t for t in sorted_tasks if t["task"] == dep_name), None)
            if dep_task and dep_task not in ordered:
                ordered.append(dep_task)
        if task not in ordered:
            ordered.append(task)
    
    # Pass 3 — Sliding window: keep tasks that fit in time budget
    time_used = 0
    final_sequence = []
    
    for task in ordered:
        if time_used + task["minutes"] <= time_available:
            time_used += task["minutes"]
            task_with_time = task.copy()
            task_with_time["cumulative_time"] = time_used
            final_sequence.append(task_with_time)
    
    return final_sequence


print("✅ Sequence ordering engine (RNN-inspired) ready.")

## Cell 6: LangChain Tools
CleanFlow uses two core tools registered with the LangChain agent.

In [ ]:
# ---------------------------------------------------------------
# LANGCHAIN TOOLS
# Tool 1: Task Analyzer — uses BERT + sequence logic to parse input
# Tool 2: Plan Formatter — turns the task list into a readable checklist
# ---------------------------------------------------------------

@tool
def analyze_cleaning_situation(user_input: str) -> str:
    """
    Analyzes the user's natural language description of their home situation.
    Uses BERT semantic embeddings to extract relevant cleaning tasks and
    RNN-inspired sequence logic to order them by urgency, room priority,
    and time constraints. Returns a structured JSON task list.
    
    Use this tool FIRST with the user's raw input to understand what needs cleaning.
    """
    # Step 1: BERT extracts semantically relevant tasks
    matched_tasks = extract_tasks_with_bert(user_input)
    
    # Step 2: Extract available time from input
    time_available = extract_time_available(user_input)
    
    # Step 3: Sequence ordering (RNN-inspired)
    ordered_tasks = sequence_order_tasks(matched_tasks, time_available)
    
    # Handle low-confidence case (vague input fallback from blueprint)
    if len(ordered_tasks) == 0:
        return json.dumps({
            "status": "needs_clarification",
            "message": "I could not identify specific cleaning tasks from this input. Please ask the user to clarify which rooms or tasks need attention.",
            "time_available": time_available
        })
    
    total_time = sum(t["minutes"] for t in ordered_tasks)
    
    return json.dumps({
        "status": "success",
        "time_available_minutes": time_available,
        "total_task_time_minutes": total_time,
        "tasks_identified": len(ordered_tasks),
        "ordered_tasks": ordered_tasks
    })


@tool
def generate_cleaning_plan(task_json: str) -> str:
    """
    Takes the structured task JSON from analyze_cleaning_situation and formats it
    into a clear, readable, time-blocked cleaning checklist for the user.
    
    Use this tool SECOND after you have the analyzed task list.
    Pass the raw JSON string from analyze_cleaning_situation as the input.
    """
    try:
        data = json.loads(task_json)
    except json.JSONDecodeError:
        return "Error: Invalid task data format. Please run analyze_cleaning_situation first."
    
    if data.get("status") == "needs_clarification":
        return data["message"]
    
    tasks = data.get("ordered_tasks", [])
    time_available = data.get("time_available_minutes", 45)
    total_time = data.get("total_task_time_minutes", 0)
    
    # Build the formatted checklist
    lines = []
    lines.append(f"🧹 YOUR CLEANFLOW PLAN ({time_available} minutes available)")
    lines.append("=" * 50)
    lines.append(f"Tasks identified: {len(tasks)} | Estimated total time: {total_time} min")
    lines.append("")
    
    time_elapsed = 0
    for i, task in enumerate(tasks, 1):
        urgency_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(task["urgency"], "⚪")
        room_clean = task["room"].title()
        time_elapsed += task["minutes"]
        lines.append(
            f"Step {i}. {urgency_emoji} [{room_clean}] {task['task'].title()} — ~{task['minutes']} min "
            f"(~{time_elapsed} min total)"
        )
    
    lines.append("")
    lines.append("=" * 50)
    
    # Motivational close
    if total_time <= time_available:
        lines.append(f"✅ You can finish everything in your {time_available}-minute window. Let's go!")
    else:
        lines.append(f"⚡ This plan fits your {time_available} minutes. High-priority tasks are listed first.")
    
    lines.append("\n💡 Tip: Start the laundry first if it's in your plan — it runs while you clean everything else!")
    
    return "\n".join(lines)


# Register tools in a list for the agent
tools = [analyze_cleaning_situation, generate_cleaning_plan]

print("✅ LangChain tools registered:")
for t in tools:
    print(f"   → {t.name}")

## Cell 7: Memory System and LangChain Agent Setup

In [ ]:
# ---------------------------------------------------------------
# MEMORY SYSTEM
# Stores conversation history across the session so the agent
# can reference previous plans and user preferences.
# Fulfills the memory requirement from the blueprint.
# ---------------------------------------------------------------

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# ---------------------------------------------------------------
# LLM SETUP
# GPT-4o-mini is cost-effective for this use case.
# Temperature 0.3 keeps responses consistent and practical.
# ---------------------------------------------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    openai_api_key=OPENAI_API_KEY
)

# ---------------------------------------------------------------
# REACT AGENT PROMPT
# ReAct (Reasoning + Acting) is the reasoning pattern from the blueprint.
# The agent reasons step by step before calling tools.
# ---------------------------------------------------------------

REACT_PROMPT_TEMPLATE = """You are CleanFlow, a smart and encouraging home cleaning assistant.
Your job is to help users turn their messy home situations into realistic, prioritized cleaning plans they can actually follow.

You have access to the following tools:
{tools}

You must always follow this ReAct reasoning loop:
Thought: Think about what the user needs and which tool to call next.
Action: The tool to use. Must be one of: {tool_names}
Action Input: The input to pass to the tool.
Observation: The result returned by the tool.
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to give a final response.
Final Answer: Your complete, friendly response to the user.

IMPORTANT RULES:
- ALWAYS call analyze_cleaning_situation FIRST with the user's exact words.
- ALWAYS call generate_cleaning_plan SECOND using the JSON output from the first tool.
- If the analysis says 'needs_clarification', ask the user a friendly follow-up question instead of generating a plan.
- Keep your Final Answer warm, encouraging, and easy to read.
- Do not make up tasks that were not in the analysis.

Conversation history so far:
{chat_history}

User's current message: {input}

{agent_scratchpad}"""

prompt = PromptTemplate(
    input_variables=["tools", "tool_names", "chat_history", "input", "agent_scratchpad"],
    template=REACT_PROMPT_TEMPLATE
)

# Build the ReAct agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# Wrap agent in executor with memory
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,           # Shows the reasoning chain (great for demo)
    handle_parsing_errors=True,
    max_iterations=6
)

print("✅ CleanFlow ReAct agent initialized with memory.")
print("   LLM: GPT-4o-mini | Framework: LangChain | Reasoning: ReAct")

## Cell 8: Run the Agent — Scenario 1
**Scenario:** User has 45 minutes and a messy kitchen + laundry situation.

In [ ]:
print("="*60)
print("SCENARIO 1: Kitchen mess + laundry, 45 minutes")
print("="*60)

response1 = agent_executor.invoke({
    "input": "My kitchen is a disaster, there are dishes everywhere and the counters are gross. "
             "I also have laundry piling up. I only have about 45 minutes before I need to leave."
})

print("\n" + "="*60)
print("CLEANFLOW FINAL ANSWER:")
print("="*60)
print(response1["output"])

## Cell 9: Run the Agent — Scenario 2
**Scenario:** Full apartment clean before guests arrive, 2 hours available.

In [ ]:
print("="*60)
print("SCENARIO 2: Pre-guest full clean, 2 hours")
print("="*60)

response2 = agent_executor.invoke({
    "input": "Guests are coming over in 2 hours and my whole apartment needs to be cleaned. "
             "The bathroom is bad, the living room has clutter everywhere, "
             "and the floors haven't been swept or vacuumed in a week."
})

print("\n" + "="*60)
print("CLEANFLOW FINAL ANSWER:")
print("="*60)
print(response2["output"])

## Cell 10: Run the Agent — Scenario 3
**Scenario:** Vague input — tests the clarification fallback from the blueprint.

In [ ]:
print("="*60)
print("SCENARIO 3: Vague input — clarification fallback test")
print("="*60)

response3 = agent_executor.invoke({
    "input": "My place is kind of a mess. I have maybe 30 minutes."
})

print("\n" + "="*60)
print("CLEANFLOW FINAL ANSWER:")
print("="*60)
print(response3["output"])

## Cell 11: Interactive Mode
Run this cell to chat with CleanFlow directly.

In [ ]:
# ---------------------------------------------------------------
# INTERACTIVE MODE
# Type your cleaning situation in plain language.
# Type 'quit' to exit.
# ---------------------------------------------------------------

print("🧹 CleanFlow is ready! Describe your home situation and I'll build your plan.")
print("Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("CleanFlow: Good luck with your cleaning! You've got this! 🧹✨")
        break
    
    try:
        response = agent_executor.invoke({"input": user_input})
        print(f"\nCleanFlow: {response['output']}\n")
    except Exception as e:
        print(f"\nCleanFlow: Sorry, I ran into an issue: {e}. Please try rephrasing.\n")

---
## Summary of Deep Learning Connections

| Component | Deep Learning Concept | Course Module |
|---|---|---|
| `SentenceTransformer('all-MiniLM-L6-v2')` | BERT-based Transformer embeddings | Transformers |
| Cosine similarity matching | Self-attention representation space | Transformers |
| `sequence_order_tasks()` — 3-pass ordering | Sequential data, time-series logic | RNNs |
| Task dependencies (hidden state carry-forward) | Hidden state in recurrent networks | RNNs |
| `ConversationBufferMemory` | Context persistence across turns | RAG / Memory |
| ReAct reasoning loop | Plan → Act → Observe cycle | Agent Reasoning |

---
*CleanFlow | ITAI 2376 Final Project | Sharise Griggs | Spring 2026*